### **1. Mount Google Drive and Setup Directory Structure**

In [1]:
from google.colab import drive
import os

# Mount Google Drive to access and save files persistently
drive.mount('/content/drive')

# Define the base directory and sub-directories in Google Drive
BASE_DIR = '/content/drive/MyDrive/finrag_project'
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

# Create the directories if they do not exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Workspace established at: {BASE_DIR}")
print("ACTION REQUIRED: Please upload your PDF financial reports into the 'data' folder inside 'finrag_project' on your Google Drive before proceeding to Cell 4.")

Mounted at /content/drive
Workspace established at: /content/drive/MyDrive/finrag_project
ACTION REQUIRED: Please upload your PDF financial reports into the 'data' folder inside 'finrag_project' on your Google Drive before proceeding to Cell 4.


### **2. Install Required Libraries**

In [2]:
!pip install -q jedi "setuptools<82"

# Install libraries for PDF parsing, embedding models, and vector database integration
!pip install -qU llama-index llama-index-readers-file llama-parse
!pip install -qU llama-index-embeddings-huggingface qdrant-client llama-index-vector-stores-qdrant

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.0/362.0 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 14.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

### **3. Configure API Keys Securely**

In [3]:
import os
from getpass import getpass

# Use getpass to securely input API credentials without hardcoding them in the notebook
print("Please enter your LlamaCloud API Key:")
os.environ["LLAMA_CLOUD_API_KEY"] = getpass()

print("Please enter your Qdrant Cluster URL (e.g., https://xxxx.qdrant.tech):")
QDRANT_URL = getpass()

print("Please enter your Qdrant API Key:")
QDRANT_API_KEY = getpass()

print("Credentials successfully loaded into environment variables.")

Please enter your LlamaCloud API Key:
··········
Please enter your Qdrant Cluster URL (e.g., https://xxxx.qdrant.tech):
··········
Please enter your Qdrant API Key:
··········
Credentials successfully loaded into environment variables.


### **4. PDF Parsing and Data Backup**

In [4]:
import os
import time
import nest_asyncio
# Apply nest_asyncio to allow nested event loops in Colab
nest_asyncio.apply()

from llama_parse import LlamaParse

# 1. Scan the directory and filter strictly for PDF files
pdf_files = [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.lower().endswith('.pdf')]
print(f"Found {len(pdf_files)} PDF files in the directory. Initiating sequential parsing...\n")

all_documents = []

# 2. Process each file with a FRESH parser instance to avoid event loop closure issues
for file_path in pdf_files:
    file_name = os.path.basename(file_path)
    print(f"Processing: {file_name}...")

    try:
        # INSTANTIATE INSIDE THE LOOP: Create a fresh parser for each file
        parser = LlamaParse(
            result_type="markdown",
            verbose=True,
            language="vi"
        )

        # Load and parse the specific file
        docs = parser.load_data(file_path)
        all_documents.extend(docs)

        print(f"   -> Success! Extracted {len(docs)} nodes from {file_name}.")

    except Exception as e:
        print(f"   -> [ERROR] Failed to parse {file_name}. Details: {e}")

    # 3. Add a small delay to prevent API rate-limiting or connection throttling
    print("   -> Resting for 3 seconds before the next file...\n")
    time.sleep(3)

print(f"PARSING PIPELINE COMPLETE! Total extracted document objects: {len(all_documents)}")

# 4. Backup the extracted data to Google Drive
parsed_file_path = os.path.join(OUTPUT_DIR, "parsed_documents.md")
with open(parsed_file_path, "w", encoding="utf-8") as f:
    for doc in all_documents:
        # Validate text payload before writing
        content = doc.text if hasattr(doc, 'text') else str(doc)
        f.write(content + "\n\n---\n\n")

print(f"Raw markdown data successfully persisted at: {parsed_file_path}")

/tmp/ipykernel_1086/3874505001.py:7: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


Found 5 PDF files in the directory. Initiating sequential parsing...

Processing: vinamilk-2025.pdf...
Started parsing the file under job_id bdeae833-c799-40dd-9dc0-f9f9a05a72bb
   -> Success! Extracted 112 nodes from vinamilk-2025.pdf.
   -> Resting for 3 seconds before the next file...

Processing: hoaphat-2025.pdf...
Started parsing the file under job_id 4439f51a-4034-4c9d-9e5b-e1b5dcb0fb0f
   -> Success! Extracted 141 nodes from hoaphat-2025.pdf.
   -> Resting for 3 seconds before the next file...

Processing: vingroup-2025.pdf...
Started parsing the file under job_id aa74df80-49fe-4645-95a5-4bde8cd35674
   -> Success! Extracted 167 nodes from vingroup-2025.pdf.
   -> Resting for 3 seconds before the next file...

Processing: fpt-2025.pdf...
Started parsing the file under job_id 31d9eed5-948c-474b-b223-0cbe3267c4be
   -> Success! Extracted 232 nodes from fpt-2025.pdf.
   -> Resting for 3 seconds before the next file...

Processing: mbbank-2025.pdf...
Started parsing the file under 

### **5. Chunking, Embedding, and Vector Upload**

In [5]:
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from llama_index.core.node_parser import MarkdownNodeParser

# 1. Configure the Embedding Model (GPU acceleration will be utilized automatically if available)
print("Initializing HuggingFace Embedding Model (BAAI/bge-m3)...")
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")
Settings.embed_model = embed_model

# 2. Chunking strategy: Use MarkdownNodeParser to prevent splitting data inside financial tables
print("Chunking documents based on markdown headers and structural elements...")
parser = MarkdownNodeParser()
nodes = parser.get_nodes_from_documents(all_documents)

# 2.1. Create a mapping of document IDs to their original metadata
doc_metadata_map = {doc.doc_id: doc.metadata for doc in all_documents}

# 2.2. Inject the missing file_name back into every node
restored_count = 0
for node in nodes:
    if node.ref_doc_id in doc_metadata_map:
        parent_metadata = doc_metadata_map[node.ref_doc_id]
        if "file_name" in parent_metadata and "file_name" not in node.metadata:
            node.metadata["file_name"] = parent_metadata["file_name"]
            restored_count += 1

print(f"Chunking completed. Generated {len(nodes)} distinct nodes.")
print(f"Successfully restored 'file_name' metadata to {restored_count} nodes!")

# 3. Establish connection to Qdrant Cloud Vector Database
print("Connecting to Qdrant Cloud...")
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

# Initialize the vector store targeting a specific collection
collection_name = "finrag_assistant_v2"
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name
)

# 4. Generate embeddings and push the vectors to Qdrant Cloud
print(f"Generating embeddings and uploading vectors to the '{collection_name}' collection...")
print("This may take a few minutes depending on the dataset size.")

storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(
    nodes=nodes,
    storage_context=storage_context,
)

print("Data Pipeline Completed Successfully! All vector representations are now stored in Qdrant Cloud.")

Initializing HuggingFace Embedding Model (BAAI/bge-m3)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Chunking documents based on markdown headers and structural elements...
Chunking completed. Generated 5680 distinct nodes.
Successfully restored 'file_name' metadata to 0 nodes!
Connecting to Qdrant Cloud...
Generating embeddings and uploading vectors to the 'finrag_assistant_v2' collection...
This may take a few minutes depending on the dataset size.
Data Pipeline Completed Successfully! All vector representations are now stored in Qdrant Cloud.
